In [1]:
import pandas as pd
import os

In [76]:
data_folder=r'..\data\transcription'
all_dfs=[]
for file in  os.listdir(data_folder):
    if file.endswith('.csv'):
        path=os.path.join(data_folder,file)
        df=pd.read_csv(path,encoding='utf-8')
        df['participant']=file #mark which participant(file)
        all_dfs.append(df.iloc[0:30,::])
data=pd.concat(all_dfs,ignore_index=True)
data.head()

,Sentence,Stimulus,Transcription Rater 1,Score,participant
0,1.0,Quiero cortarme el pelo (7),Quiero cortarme el pelo,NaN,participant1.csv
1,2.0,El libro está en la mesa (7),El libro está en la mesa,NaN,participant1.csv
2,3.0,El carro lo tiene Pedro (8),El carro lo tiene Pedro,NaN,participant1.csv
3,4.0,El se ducha cada mañana (9),El se ducha cada mañana,NaN,participant1.csv
4,5.0,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,NaN,participant1.csv


In [60]:
data.shape,data.columns

((120, 5),
 Index(['Sentence', 'Stimulus', 'Transcription Rater 1', 'Score',
        'participant'],
       dtype='object'))

In [61]:
from sentence_transformers import SentenceTransformer,util
import re

In [62]:
def clean(text):
    return re.sub(r"\s*\(.*?\)","",str(text)).strip()

In [63]:
data['Stimulus']=data['Stimulus'].apply(clean)

In [64]:
data.head()

,Sentence,Stimulus,Transcription Rater 1,Score,participant
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo,NaN,participant1.csv
1,2.0,El libro está en la mesa,El libro está en la mesa,NaN,participant1.csv
2,3.0,El carro lo tiene Pedro,El carro lo tiene Pedro,NaN,participant1.csv
3,4.0,El se ducha cada mañana,El se ducha cada mañana,NaN,participant1.csv
4,5.0,¿Qué dice usted que va a hacer hoy?,Que dices ustedes se que van a hacer hoy?,NaN,participant1.csv


In [65]:
model=SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
#test  

target = data["Stimulus"].iloc[6]
response = data["Transcription Rater 1"].iloc[6]
emb1 = model.encode(target)
emb2 = model.encode(response)
similarity = util.cos_sim(emb1, emb2)

print("Target:", target)
print("Response:", response)
print("Similarity:", similarity.item())

Target: Las calles de esta ciudad son muy anchas
Response: Las calles de esta cuidad son muy anchas
Similarity: 0.9566935300827026


In [67]:
def compute_similarity(target,response):
    emb1=model.encode(target)
    emb2 = model.encode(response)
    sim = util.cos_sim(emb1, emb2).item()
    return sim

In [68]:
data['Similarity']=data.apply(lambda r: compute_similarity(r["Stimulus"],
                                                           r["Transcription Rater 1"]),axis=1)

In [69]:
data[["Stimulus","Transcription Rater 1","Similarity"]].head(10)

,Stimulus,Transcription Rater 1,Similarity
0,Quiero cortarme el pelo,Quiero cortarme el pelo,1.000000
1,El libro está en la mesa,El libro está en la mesa,1.000000
2,El carro lo tiene Pedro,El carro lo tiene Pedro,1.000000
3,El se ducha cada mañana,El se ducha cada mañana,1.000000
4,¿Qué dice usted que va a hacer hoy?,Que dices ustedes se que van a hacer hoy?,0.954708
5,Dudo que sepa manejar muy bien,Dudo que sepa manajar bien,0.854764
6,Las calles de esta ciudad son muy anchas,Las calles de esta cuidad son muy anchas,0.956694
7,Puede que llueva mañana todo el día,Puede que lleva mañana todo el día,0.749310
8,Las casas son muy bonitas pero caras,Las casas son muy bonitas pero muy cadas,0.873150
9,Me gustan las películas que acaban bien,Me gustan las peliculas que acaban bien,0.540314


In [71]:
def assign_score(sim):
    
    if sim >= 0.85:
        return 2      # correct meaning
    
    elif sim >= 0.65:
        return 1      # partially correct
    
    else:
        return 0      # incorrect meaning

In [72]:
data["Predicted_score"] = data["Similarity"].apply(assign_score)

In [73]:
data.head(10)

,Sentence,Stimulus,Transcription Rater 1,Score,participant,Similarity,Predicted_score
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo,NaN,participant1.csv,1.000000,2
1,2.0,El libro está en la mesa,El libro está en la mesa,NaN,participant1.csv,1.000000,2
2,3.0,El carro lo tiene Pedro,El carro lo tiene Pedro,NaN,participant1.csv,1.000000,2
3,4.0,El se ducha cada mañana,El se ducha cada mañana,NaN,participant1.csv,1.000000,2
4,5.0,¿Qué dice usted que va a hacer hoy?,Que dices ustedes se que van a hacer hoy?,NaN,participant1.csv,0.954708,2
5,6.0,Dudo que sepa manejar muy bien,Dudo que sepa manajar bien,NaN,participant1.csv,0.854764,2
6,7.0,Las calles de esta ciudad son muy anchas,Las calles de esta cuidad son muy anchas,NaN,participant1.csv,0.956694,2
7,8.0,Puede que llueva mañana todo el día,Puede que lleva mañana todo el día,NaN,participant1.csv,0.749310,1
8,9.0,Las casas son muy bonitas pero caras,Las casas son muy bonitas pero muy cadas,NaN,participant1.csv,0.873150,2
9,10.0,Me gustan las películas que acaban bien,Me gustan las peliculas que acaban bien,NaN,participant1.csv,0.540314,0


In [74]:
data.to_csv("../results/AutoEIT_scored_output.csv", index=False)